In [26]:
# ==========================================================
# Task 3: Feature Engineering - Event Impact Model
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
import os



# ==========================================================
# Load Data
# ==========================================================

data_file = "../data/raw/ethiopia_fi_unified_data.csv"
impact_file = "../data/processed/new_impact_links.csv"


data = pd.read_csv(data_file)
impact = pd.read_csv(impact_file)


print("="*60)
print("DATA LOADED")
print("="*60)

print("Data shape:", data.shape)
print("Impact shape:", impact.shape)



# ==========================================================
# Date Cleaning
# ==========================================================

data["observation_date"] = pd.to_datetime(
    data["observation_date"],
    errors="coerce"
)



# ==========================================================
# Split Events and Observations
# ==========================================================

events = data[
    data["record_type"]
    .astype(str)
    .str.lower()
    ==
    "event"
].copy()



observations = data[
    data["record_type"]
    .astype(str)
    .str.lower()
    ==
    "observation"
].copy()



# Event date

events["event_date"] = pd.to_datetime(
    events["observation_date"],
    errors="coerce"
)



print("\nEvents:", len(events))
print("Observations:", len(observations))



# ==========================================================
# Clean Impact Magnitude
# ==========================================================

print("\nOriginal impact magnitude:")
print(
    impact["impact_magnitude"].unique()
)



# Convert numeric values

impact["impact_magnitude"] = pd.to_numeric(
    impact["impact_magnitude"],
    errors="coerce"
)



# Convert text values if needed

if impact["impact_magnitude"].isna().all():

    magnitude_map = {

        "very high": 3,
        "high": 2,
        "medium": 1,
        "moderate": 1,
        "low": 0.5,
        "very low": 0.2,

        "positive": 1,
        "negative": -1
    }


    impact["impact_magnitude"] = (
        impact["impact_magnitude"]
        .astype(str)
        .str.lower()
        .map(magnitude_map)
    )



print("\nCleaned magnitude:")
print(
    impact["impact_magnitude"].unique()
)



# ==========================================================
# Merge Impact Links With Events
# ==========================================================


event_info = events[
    [
        "record_id",
        "category",
        "pillar",
        "event_date"
    ]
]


impact_df = impact.merge(
    event_info,
    left_on="parent_id",
    right_on="record_id",
    how="left"
)



print("\nMerged data:")
print(impact_df.shape)



print(
    impact_df[
        [
            "parent_id",
            "category",
            "related_indicator",
            "impact_magnitude",
            "event_date"
        ]
    ].head()
)



# ==========================================================
# Event Indicator Association Matrix
# ==========================================================


association = impact_df.pivot_table(
    index="category",
    columns="related_indicator",
    values="impact_magnitude",
    aggfunc="mean"
)



print("\nAssociation Matrix Shape:")
print(association.shape)



if not association.empty:


    print(association)


    plt.figure(figsize=(12,6))


    sns.heatmap(
        association,
        annot=True,
        cmap="RdYlGn",
        center=0
    )


    plt.title(
        "Event Indicator Association Matrix"
    )

    plt.tight_layout()

    plt.show()


else:

    print(
        "Association matrix is empty. Check indicator/magnitude data."
    )



# ==========================================================
# Event Effect Function
# ==========================================================


def event_effect(
    obs_date,
    event_date,
    magnitude,
    lag
):

    if pd.isna(event_date):
        return 0


    if pd.isna(magnitude):
        return 0


    activation = (
        event_date
        +
        pd.DateOffset(
            months=int(lag)
        )
    )


    if obs_date >= activation:

        return magnitude


    return 0




# ==========================================================
# Prediction Generation
# ==========================================================


results = []



available_indicators = (
    impact_df["related_indicator"]
    .dropna()
    .unique()
)



print("\nIndicators found:")
print(len(available_indicators))



for indicator in available_indicators:


    obs = observations[
        observations["indicator_code"]
        ==
        indicator
    ].copy()



    if obs.empty:

        continue



    obs = obs.sort_values(
        "observation_date"
    )



    links = impact_df[
        impact_df["related_indicator"]
        ==
        indicator
    ]



    predictions = []



    for date in obs["observation_date"]:


        total_effect = 0



        for _, link in links.iterrows():


            effect = event_effect(
                date,
                link["event_date"],
                link["impact_magnitude"],
                link["lag_months"]
            )



            if (
                str(link["impact_direction"])
                .lower()
                ==
                "negative"
            ):

                effect = -abs(effect)



            total_effect += effect



        predictions.append(
            total_effect
        )



    obs["predicted_effect"] = predictions

    obs["indicator"] = indicator


    results.append(obs)



# ==========================================================
# Validation
# ==========================================================


validation = []



if len(results) > 0:


    results = pd.concat(
        results,
        ignore_index=True
    )


    print("\nValidation")


    for indicator in results["indicator"].unique():


        temp = results[
            results["indicator"]
            ==
            indicator
        ]


        mae = mean_absolute_error(
            temp["value_numeric"],
            temp["predicted_effect"]
        )


        validation.append(
            [
                indicator,
                mae
            ]
        )



        plt.figure(figsize=(8,4))


        plt.plot(
            temp["observation_date"],
            temp["value_numeric"],
            marker="o",
            label="Observed"
        )


        plt.plot(
            temp["observation_date"],
            temp["predicted_effect"],
            marker="s",
            label="Predicted Effect"
        )


        plt.title(indicator)

        plt.xlabel("Date")

        plt.ylabel("Value")

        plt.legend()

        plt.grid(True)

        plt.show()



    validation = pd.DataFrame(
        validation,
        columns=[
            "Indicator",
            "MAE"
        ]
    )


    print(validation)



else:

    print(
        "No matching indicators between events and observations."
    )



# ==========================================================
# Save Outputs
# ==========================================================


os.makedirs(
    "../reports",
    exist_ok=True
)



association.to_csv(
    "../reports/event_indicator_association_matrix.csv"
)



if len(validation) > 0:

    validation.to_csv(
        "../reports/impact_validation.csv",
        index=False
    )



print("\nSaved:")
print(
    "../reports/event_indicator_association_matrix.csv"
)


if len(validation) > 0:

    print(
        "../reports/impact_validation.csv"
    )



# ==========================================================
# Assumptions
# ==========================================================

print("\nMODEL ASSUMPTIONS")

print("- Event effects start after lag months.")
print("- Effects are additive.")
print("- Impact magnitude remains constant.")
print("- Positive and negative impacts are separated.")
print("- No decay applied.")
print("- External factors are not modeled.")

DATA LOADED
Data shape: (43, 34)
Impact shape: (1, 7)

Events: 10
Observations: 30

Original impact magnitude:
<StringArray>
['high']
Length: 1, dtype: str

Cleaned magnitude:
[nan]

Merged data:
(1, 11)
                     parent_id category    related_indicator  \
0  policy_2021_digital_finance      NaN  MOBILE_MONEY_ACCESS   

   impact_magnitude event_date  
0               NaN        NaT  

Association Matrix Shape:
(0, 0)
Association matrix is empty. Check indicator/magnitude data.

Indicators found:
1
No matching indicators between events and observations.

Saved:
../reports/event_indicator_association_matrix.csv

MODEL ASSUMPTIONS
- Event effects start after lag months.
- Effects are additive.
- Impact magnitude remains constant.
- Positive and negative impacts are separated.
- No decay applied.
- External factors are not modeled.
